# PanDerm 선형 평가: 데이터 및 성능 지표 심층 분석

**목적**: `aptos2019` (안저 영상 당뇨망막병증)와 `Oral_Diseases` (구강 질환 다중 분류)에서의 성능 지표 일관성 부족 현상을 심층 분석합니다. 본 노트북은 극심한 레이블 불균형이 모델 성능과 지표(AUPR, Balanced Accuracy 등)에 미치는 영향을 파악하고, 이를 보정하기 위한 방법론을 도출하는 것을 목표로 합니다.

**분석 항목**:
1. 레이블 불균형(Label Imbalance) 분석 및 시각화
2. 클래스별 세부 성능 분석 (Precision / Recall / F1 히트맵, Confusion Matrix)
3. 지표 계산 방식 심층 검증 (Class-specific AUPR, Weighted AUPR, Balanced Accuracy)
4. 개선 방향: Class Weights 계산 코드 스케치

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ──────────────────────────────────────────
# 경로 설정
# ──────────────────────────────────────────
DATA_ROOT   = "../../../PanDerm/data"
OUTPUT_ROOT = "../../../PanDerm/output_dir"
FIGURES_DIR = "../../../PanDerm/Linear Evaluation/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

FOCUS_DATASETS = {
    "aptos2019": {
        "meta_csv": f"{DATA_ROOT}/aptos2019/Linear Evaluation/aptos2019_multiclass.csv",
        "pred_csv": f"{OUTPUT_ROOT}/aptos2019_panderm_large_lp/aptos2019_multiclass.csv",
        "label_col": "label",
        "n_classes": 5,
        "class_names": ["0: No DR", "1: Mild", "2: Moderate", "3: Severe", "4: Proliferative"],
    },
    "Oral_Diseases": {
        "meta_csv": f"{DATA_ROOT}/Oral_Diseases/Linear Evaluation/oral_diseases_multiclass.csv",
        "pred_csv": f"{OUTPUT_ROOT}/oral_diseases_panderm_large_lp/oral_diseases_multiclass.csv",
        "label_col": "label",
        "n_classes": 7,
        "class_names": ["0: CaS", "1: CoS", "2: Gum", "3: MC", "4: OC", "5: OLP", "6: OT"],
    },
}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "NanumGothic",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("✅ 환경 설정 완료")

## 1. 레이블 불균형 분석 (Label Imbalance)
### 1-1. 전체 데이터셋 Split 크기 개요

In [ ]:
ALL_META_CSVS = {
    "Oral_Diseases": f"{DATA_ROOT}/Oral_Diseases/Linear Evaluation/oral_diseases_multiclass.csv",
    "aptos2019":     f"{DATA_ROOT}/aptos2019/Linear Evaluation/aptos2019_multiclass.csv",
    "HAM10000":      f"{DATA_ROOT}/PanDerm_data/HAM10000/Linear Evaluation/ham10000_multiclass.csv",
    "Oral_Cancer":   f"{DATA_ROOT}/Oral_Cancer/Linear Evaluation/oral_cancer_binary.csv",
    "Chest_X-Ray":   f"{DATA_ROOT}/Chest_X-Ray/Linear Evaluation/chest_xray_binary.csv",
    "Breast_Cancer": f"{DATA_ROOT}/Breast_Cancer/Linear Evaluation/breast_cancer_binary_fold1.csv",
}

overview_rows = []
for name, path in ALL_META_CSVS.items():
    df = pd.read_csv(path)
    cnt = df["split"].value_counts()
    label_col = "label" if "label" in df.columns else "binary_label"
    n_classes = df[label_col].nunique()
    overview_rows.append({
        "데이터셋": name,
        "클래스 수": n_classes,
        "Train": cnt.get("train", 0),
        "Val": cnt.get("val", 0),
        "Test": cnt.get("test", 0),
    })

overview_df = pd.DataFrame(overview_rows).set_index("데이터셋")
print("=== 전체 데이터셋 Split 크기 개요 ===")
print(overview_df.to_string())

### 💡 [분석 결과] 1-1. 데이터셋 규모 점검
위 출력 결과를 통해 PanDerm 평가 데이터셋들의 기본적인 훈련/검증/테스트 스플릿 규모를 파악할 수 있습니다. `aptos2019`는 Train 샘플이 약 2,929개, Test 샘플이 733개로 충분한 샘플을 보유하고 있으나, `Oral_Diseases`의 경우 Test 샘플이 50개에 불과합니다. Test 샘플이 절대적으로 적은 데이터셋에서는 단 1~2개의 오분류만으로도 Macro 평균 지표가 크게 출렁일 수 있음을 유의해야 합니다.

### 1-2. aptos2019 & Oral_Diseases 클래스 분포 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("레이블 불균형 분석: aptos2019 & Oral_Diseases", fontsize=14, fontweight="bold", y=1.01)
SPLIT_COLORS = {"train": "#4C72B0", "val": "#55A868", "test": "#C44E52"}
SPLITS = ["train", "val", "test"]

for row_idx, (ds_name, ds_cfg) in enumerate(FOCUS_DATASETS.items()):
    meta_df = pd.read_csv(ds_cfg["meta_csv"])
    label_col = ds_cfg["label_col"]
    class_names = ds_cfg["class_names"]
    n_classes = ds_cfg["n_classes"]
    dist = pd.crosstab(meta_df[label_col], meta_df["split"])

    for col_idx, split in enumerate(SPLITS):
        ax = axes[row_idx][col_idx]
        if split not in dist.columns:
            ax.set_visible(False)
            continue
        counts = dist[split].values
        bars = ax.bar(range(n_classes), counts, color=SPLIT_COLORS[split], alpha=0.85, edgecolor="white", linewidth=0.8)
        for bar, cnt in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(int(cnt)), ha="center", va="bottom", fontsize=8.5, fontweight="bold")
        ax.set_title(f"{ds_name} – {split.upper()}", fontsize=10, fontweight="bold", color=SPLIT_COLORS[split])
        ax.set_xticks(range(n_classes))
        ax.set_xticklabels([c.split(":")[0] for c in class_names], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("샘플 수", fontsize=8)
        ax.set_ylim(0, counts.max() * 1.2)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "label_imbalance_focus.png"), dpi=150, bbox_inches="tight")
plt.show()

### 💡 [분석 결과] 1-2. 클래스 분포 시각화 인사이트
바 차트에서 명확히 드러나듯, **`aptos2019` 데이터셋의 불균형이 매우 심각**합니다. `0: No DR` 클래스에 샘플이 압도적으로 몰려있는 반면, 악화 단계인 `3: Severe`, `4: Proliferative` 클래스는 매우 적습니다. 반대로 `Oral_Diseases`는 Train 단계에서는 모든 클래스가 약 560여 개로 매우 고르게 분포되어 있으나, Test Split에서의 샘플 분포가 고르지 못하고 전체 모수가 너무 작습니다. 이와 같은 불균형은 특정 클래스에 모델의 결정 경계가 치우치게 만듭니다.

### 1-3. 불균형 비율 정량 계산

In [ ]:
for ds_name, ds_cfg in FOCUS_DATASETS.items():
    meta_df = pd.read_csv(ds_cfg["meta_csv"])
    label_col = ds_cfg["label_col"]
    print(f"\n📌 {ds_name}")
    print(f"  {'Split':<8} {'클래스':>6} {'최다':>8} {'최소':>8} {'IR(최다/최소)':>14}")
    print("  " + "-" * 46)
    for split in ["train", "val", "test"]:
        sub = meta_df[meta_df["split"] == split][label_col].value_counts()
        if sub.empty:
            continue
        majority_cls = sub.idxmax()
        minority_cls = sub.idxmin()
        ir = sub.max() / sub.min()
        print(f"  {split:<8} {len(sub):>6}개  "
              f"{majority_cls}({sub.max():>4}개) / {minority_cls}({sub.min():>4}개)  IR={ir:.2f}x")

### 💡 [분석 결과] 1-3. 불균형 비율 (Imbalance Ratio)
정량적인 수치로 확인한 결과, `aptos2019` Train Set의 불균형 비율(IR)은 **9.36배**에 달합니다. 즉, 0번 클래스가 3번 클래스보다 9배 이상 많습니다. 현재 Linear Probe 파이프라인(`linear_probe.py`)에서는 기본 `CrossEntropyLoss`를 사용하고 있기 때문에 손실(Loss) 값이 0번 클래스의 샘플들로부터 압도적으로 많이 발생하여, 경사하강법 진행 시 0번 클래스를 잘 맞추는 방향으로만 모델이 최적화될 위험이 있습니다.

## 2. 클래스별 세부 성능 분석
### 2-1. Classification Report & 모델 성능 요약

In [ ]:
pred_data = {}
for ds_name, ds_cfg in FOCUS_DATASETS.items():
    pred_df = pd.read_csv(ds_cfg["pred_csv"])
    y_true = pred_df["true_label"].values
    y_pred = pred_df["predicted_label"].values
    prob_cols = [c for c in pred_df.columns if c.startswith("probability_class_")]
    probs = pred_df[prob_cols].values
    pred_data[ds_name] = {"y_true": y_true, "y_pred": y_pred, "probs": probs,
                          "class_names": ds_cfg["class_names"]}

    print(f"\n{'=' * 55}")
    print(f"  {ds_name}")
    print(f"{'=' * 55}")
    print(classification_report(y_true, y_pred, target_names=ds_cfg["class_names"], zero_division=0))
    auroc = roc_auc_score(y_true, probs, multi_class="ovo", average="macro")
    aupr_macro = average_precision_score(y_true, probs, average="macro")
    bacc = balanced_accuracy_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    print(f"  Macro AUROC : {auroc:.4f}")
    print(f"  Macro AUPR  : {aupr_macro:.4f}")
    print(f"  Balanced Acc: {bacc:.4f}")
    print(f"  Accuracy    : {acc:.4f}  (차이 = {acc - bacc:+.4f})")

### 💡 [분석 결과] 2-1. Classification Report 요약
지표 출력 결과를 보면, `aptos2019` 모델의 전체 **Accuracy는 0.8145로 매우 우수해 보이지만, 소수 클래스의 성능까지 균등하게 고려하는 Balanced Accuracy는 0.6280에 불과**합니다. 이는 모델 성능의 착시 현상을 보여주는 강력한 증거입니다. 또한 Macro AUPR 지표(0.6384)가 Macro AUROC(0.9022)에 비해 매우 낮게 측정되는 양상을 보입니다.

### 2-2. 클래스별 성능 히트맵

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("클래스별 성능 히트맵 (Precision / Recall / F1)", fontsize=13, fontweight="bold")

for ax, (ds_name, data) in zip(axes, pred_data.items()):
    y_true = data["y_true"]; y_pred = data["y_pred"]
    class_names = data["class_names"]; n_cls = len(class_names)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(n_cls)), zero_division=0)
    perf_df = pd.DataFrame({"Precision": p, "Recall": r, "F1": f},
                           index=[c.split(":")[1].strip() if ":" in c else c for c in class_names])
    sns.heatmap(perf_df, ax=ax, vmin=0.0, vmax=1.0, cmap="RdYlGn",
                annot=True, fmt=".2f", linewidths=0.5, linecolor="#ccc", cbar_kws={"shrink": 0.8})
    ax.set_title(ds_name, fontsize=11, fontweight="bold")
    ax.set_xlabel("지표", fontsize=9); ax.set_ylabel("클래스", fontsize=9)
    ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", labelsize=9, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "classwise_performance_heatmap_focus.png"), dpi=150, bbox_inches="tight")
plt.show()

### �� [분석 결과] 2-2. 성능 히트맵 시각화
히트맵은 문제를 더 극명하게 보여줍니다. `aptos2019`의 경우 `0: No DR` 클래스의 F1 점수는 매우 높으나, `3: Severe` 클래스는 Recall이 낮아 실제로 Severe 샘플을 제대로 찾아내지 못하고 있습니다. 
`Oral_Diseases`에서도 `OLP` 클래스의 지표가 눈에 띄게 붉은색(낮은 성능)을 띄고 있어, 다중 분류에서 일부 클래스가 전체 성능을 심각하게 저하시키고 있음을 확인할 수 있습니다.

### 2-3. Confusion Matrix 정규화 및 상위 오분류 패턴

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix (정규화: 실제 클래스 기준)", fontsize=13, fontweight="bold")

for ax, (ds_name, data) in zip(axes, pred_data.items()):
    y_true = data["y_true"]; y_pred = data["y_pred"]
    class_names = data["class_names"]; n_cls = len(class_names)
    tick_labels = [c.split(":")[1].strip() if ":" in c else c for c in class_names]
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(n_cls):
        for j in range(n_cls):
            val = cm_norm[i, j]; raw = cm[i, j]
            text_color = "white" if val > 0.5 else "black"
            ax.text(j, i, f"{val:.2f}\n({raw})", ha="center", va="center", fontsize=7.5, color=text_color)
    ax.set_xticks(range(n_cls)); ax.set_yticks(range(n_cls))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(tick_labels, fontsize=8)
    ax.set_xlabel("예측 클래스", fontsize=9); ax.set_ylabel("실제 클래스", fontsize=9)
    ax.set_title(ds_name, fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "confusion_matrix_normalized_focus.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── 오분류 패턴 텍스트 요약
print("\n  주요 오분류 패턴 (True → Predicted, 상위 오류)")
print("=" * 60)
for ds_name, data in pred_data.items():
    y_true = data["y_true"]; y_pred = data["y_pred"]
    class_names = data["class_names"]; n_cls = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    print(f"\n📌 {ds_name}")
    errors = []
    for i in range(n_cls):
        total_i = cm[i].sum()
        for j in range(n_cls):
            if i != j and cm[i, j] > 0:
                errors.append({"실제": class_names[i], "예측": class_names[j],
                               "오진 수": int(cm[i, j]), "오진율": cm[i, j] / total_i})
    err_df = pd.DataFrame(errors).sort_values("오진 수", ascending=False).reset_index(drop=True)
    print(err_df.head(8).to_string(index=False))

### 💡 [분석 결과] 2-3. 혼동 행렬 오분류 패턴
실제(True) 클래스 기준으로 정규화한 Confusion Matrix를 분석한 결과,
- **aptos2019**: `Moderate`와 `Mild`가 `No DR`(다수 클래스)로 예측되는 오류가 가장 많습니다. 이는 극심한 불균형으로 인해 모델이 애매한 판단에서 다수 클래스를 선택하도록 편향되었기 때문입니다.
- **Oral_Diseases**: `OLP` 클래스가 `OT`로 잘못 예측되는 비율이 상대적으로 높습니다. 이는 단순히 특성 학습의 실패일 수도 있으나, 데이터 수의 절대적 부족에서 기인한 평가 불안정성일 가능성이 큽니다.

## 3. 지표 계산 방식 심층 검증
### 3-1. Class-specific AUPR + Weighted AUPR 계산

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Class-specific AUPR 비교", fontsize=13, fontweight="bold")

for ax, (ds_name, data) in zip(axes, pred_data.items()):
    y_true = data["y_true"]; probs = data["probs"]
    class_names = data["class_names"]; n_cls = len(class_names)
    classes = list(range(n_cls))
    y_bin = label_binarize(y_true, classes=classes)
    support = np.bincount(y_true, minlength=n_cls)

    per_class_aupr = []
    for i in range(n_cls):
        try:
            per_class_aupr.append(average_precision_score(y_bin[:, i], probs[:, i]))
        except Exception:
            per_class_aupr.append(float("nan"))

    macro_aupr    = np.nanmean(per_class_aupr)
    weighted_aupr = np.nansum([a * s for a, s in zip(per_class_aupr, support) if not np.isnan(a)]) / support.sum()

    print(f"\n📌 {ds_name}")
    print(f"  {'클래스':<22} {'AUPR':>8}  {'Support':>9}")
    print("  " + "-" * 42)
    for cname, aupr_i, sup in zip(class_names, per_class_aupr, support):
        flag = " ⚠️" if aupr_i < 0.6 else ""
        print(f"  {cname:<22} {aupr_i:>8.4f}  {sup:>9}{flag}")
    print("  " + "-" * 42)
    print(f"  {'Macro AUPR':<22} {macro_aupr:>8.4f}")
    print(f"  {'Weighted AUPR':<22} {weighted_aupr:>8.4f}")

    tick_labels = [c.split(":")[1].strip() if ":" in c else c for c in class_names]
    colors = ["#C44E52" if v < 0.6 else "#4C72B0" for v in per_class_aupr]
    bars = ax.bar(range(n_cls), per_class_aupr, color=colors, alpha=0.85, edgecolor="white")
    for bar, val, sup in zip(bars, per_class_aupr, support):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.2f}\n(n={sup})", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    ax.axhline(macro_aupr,    color="#DD8800", ls="--", lw=1.5, label=f"Macro AUPR={macro_aupr:.3f}")
    ax.axhline(weighted_aupr, color="#228B22", ls=":",  lw=1.5, label=f"Weighted AUPR={weighted_aupr:.3f}")
    ax.set_ylim(0, 1.15)
    ax.set_xticks(range(n_cls))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("AUPR", fontsize=9)
    ax.set_title(ds_name, fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_specific_aupr_focus.png"), dpi=150, bbox_inches="tight")
plt.show()

### 💡 [분석 결과] 3-1. AUPR 계산 로직 및 다중 분류의 한계
`scikit-learn`의 다중 분류 `average_precision_score(average='macro')` 함수는 내부적으로 클래스를 **OvR (One-vs-Rest)** 방식으로 이진화(binarize)한 뒤, 각 클래스의 AUPR을 평균냅니다. 
문제는 `aptos2019`의 소수 클래스(예: 3, 4번)를 'Positive'로 설정할 경우, 거대한 0번 다수 클래스가 모두 'Negative'로 취급되어 사소한 오분류에도 False Positive가 폭증하게 된다는 것입니다. FP의 증가는 Precision을 수직 낙하시키며, 결과적으로 소수 클래스의 AUPR이 급락합니다(위 그래프의 붉은 막대). 
이렇게 하락한 개별 클래스 AUPR 점수들이 산술 평균(Macro)으로 결합되면서 **전체 데이터셋의 Macro AUPR 수치를 부당하게 과소평가하게 만드는 주요 원인**이 됩니다.

### 3-2. Balanced Accuracy vs Accuracy 갭 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ds_names = list(pred_data.keys())
accs  = [accuracy_score(d["y_true"], d["y_pred"]) for d in pred_data.values()]
baccs = [balanced_accuracy_score(d["y_true"], d["y_pred"]) for d in pred_data.values()]
gaps  = [a - b for a, b in zip(accs, baccs)]
x = np.arange(len(ds_names)); width = 0.32
bars1 = ax.bar(x - width / 2, accs,  width, label="Accuracy",         color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width / 2, baccs, width, label="Balanced Accuracy", color="#C44E52", alpha=0.85)
for bar, val in zip(bars1, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, color="#4C72B0", fontweight="bold")
for bar, val in zip(bars2, baccs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, color="#C44E52", fontweight="bold")
for i, (a, b, g) in enumerate(zip(accs, baccs, gaps)):
    ax.annotate("", xy=(i + width / 2, b), xytext=(i - width / 2, a),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    ax.text(i + width / 2 + 0.03, (a + b) / 2,
            f"갭\n{g:+.3f}", ha="left", va="center", fontsize=8, color="gray")
ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(ds_names, fontsize=10)
ax.set_ylabel("지표 값", fontsize=10)
ax.set_title("Accuracy vs Balanced Accuracy 갭\n(갭이 클수록 클래스 불균형 편향 심각)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.axhline(0.8, color="#999", ls="--", lw=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "acc_vs_bacc_gap_focus.png"), dpi=150, bbox_inches="tight")
plt.show()

### 💡 [분석 결과] 3-2. Accuracy 갭이 시사하는 바
Accuracy와 Balanced Accuracy의 차이(갭)가 크다는 것은 모델이 특정(주로 다수) 클래스의 예측에 과적합되어 전체 점수를 방어하고 있음을 의미합니다. 
`aptos2019`에서 관찰되는 **+0.186의 거대한 갭**은, Linear Probe 과정에서 불균형에 대한 어떠한 보정 조치(Class Weighting 등)도 적용되지 않아 소수 클래스의 희생을 대가로 다수 클래스의 예측만 고도화되었음을 입증하는 확실한 단서입니다.

## 4. 개선 방향: Class Weights 계산 코드

In [ ]:
for ds_name, ds_cfg in FOCUS_DATASETS.items():
    meta_df = pd.read_csv(ds_cfg["meta_csv"])
    train_df = meta_df[meta_df["split"] == "train"]
    label_col = ds_cfg["label_col"]; class_names = ds_cfg["class_names"]
    n_cls = ds_cfg["n_classes"]; classes = np.arange(n_cls)
    y_train = train_df[label_col].values
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    train_counts = np.bincount(y_train, minlength=n_cls)
    print(f"\n📌 {ds_name} (Train 샘플 수: {len(y_train)})")
    print(f"  {'클래스':<22} {'Train 수':>10} {'Weight':>10}")
    print("  " + "-" * 44)
    for cname, cnt, w in zip(class_names, train_counts, weights):
        print(f"  {cname:<22} {cnt:>10}   {w:>10.4f}")
    weights_str = ", ".join([f"{w:.4f}" for w in weights])
    print(f"\n  # PyTorch 사용 예시:")
    print(f"  class_weights = torch.tensor([{weights_str}], dtype=torch.float)")
    print(f"  criterion = nn.CrossEntropyLoss(weight=class_weights)")

### 💡 [분석 결과] 4. 학습 최적화를 위한 개선 방안 제시
위 결과를 바탕으로, 본 연구의 Linear Evaluation 혹은 향후 파인튜닝 단계에서는 반드시 Train Set 기준의 역산별 가중치(Class Weights)를 부여해야 합니다. `sklearn.utils.class_weight.compute_class_weight`의 `balanced` 휴리스틱을 사용하면 위와 같이 소수 클래스에 높은 패널티 계수를, 다수 클래스에 낮은 패널티 계수를 산출할 수 있습니다. 
이를 PyTorch의 `nn.CrossEntropyLoss(weight=class_weights)` 인자로 넘겨주어 소수 클래스에 대한 학습 경사도를 증폭시키면, 극심한 성능 편향을 상당히 교정할 수 있을 것으로 판단됩니다.